In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")

In [9]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [10]:
docs = [doc1,doc2,doc3,doc4,doc5]

In [11]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(),
    #location of the folder where embeding vector stores
    persist_directory='chroma_db', 
    #collection name
    collection_name='sample'
)

In [12]:
#add document
vector_store.add_documents(docs)

['874cfe20-65f0-44bd-8efe-7e9d1baf845c',
 'a4901e55-c972-424c-be7f-80086d2d68c3',
 'dabf3130-4570-4fef-86f1-d28b71c421f3',
 '25ba1dc0-23c3-4943-b7fd-ba797e8dcf11',
 'b326fdd0-51b2-40a4-a312-bac271920cc5']

In [14]:
#view document
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['874cfe20-65f0-44bd-8efe-7e9d1baf845c',
  'a4901e55-c972-424c-be7f-80086d2d68c3',
  'dabf3130-4570-4fef-86f1-d28b71c421f3',
  '25ba1dc0-23c3-4943-b7fd-ba797e8dcf11',
  'b326fdd0-51b2-40a4-a312-bac271920cc5'],
 'embeddings': array([[-0.00210453, -0.00214285,  0.0268    , ..., -0.01707893,
         -0.00366616,  0.01357884],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(5, 1536)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [20]:
vector_store.similarity_search(
    query='who among these are bowler?',
    k=1
)

[Document(id='25ba1dc0-23c3-4943-b7fd-ba797e8dcf11', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [ ]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are bowler?',
    k=3
)

[(Document(id='25ba1dc0-23c3-4943-b7fd-ba797e8dcf11', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.3638073205947876),
 (Document(id='b326fdd0-51b2-40a4-a312-bac271920cc5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.42706960439682007),
 (Document(id='874cfe20-65f0-44bd-8efe-7e9d1baf845c', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
  0.44979649782180786)]

In [24]:
#meta-data filtering
vector_store.similarity_search_with_score(
    query='',
    filter={"team":"Chennai Super Kings"}
)

[(Document(id='dabf3130-4570-4fef-86f1-d28b71c421f3', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.6488258242607117),
 (Document(id='b326fdd0-51b2-40a4-a312-bac271920cc5', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6566494703292847)]

In [26]:
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of RCB is remowed for his consistent batting",
    metadata={"team":"RCB"}
)
vector_store.update_document(document_id = '874cfe20-65f0-44bd-8efe-7e9d1baf845c', document=updated_doc1)

In [27]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['874cfe20-65f0-44bd-8efe-7e9d1baf845c',
  'a4901e55-c972-424c-be7f-80086d2d68c3',
  'dabf3130-4570-4fef-86f1-d28b71c421f3',
  '25ba1dc0-23c3-4943-b7fd-ba797e8dcf11',
  'b326fdd0-51b2-40a4-a312-bac271920cc5'],
 'embeddings': array([[-0.00519968, -0.00933361,  0.024135  , ..., -0.00099046,
          0.00286793,  0.01342452],
        [-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(5, 1536)),
 'documents': ['Virat Kohli, the former captain of RCB is remowed for his consistent batting',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and abili

In [ ]:
#delete document
vector_store.delete(ids=['874cfe20-65f0-44bd-8efe-7e9d1baf845c'])

In [29]:
#view document
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['a4901e55-c972-424c-be7f-80086d2d68c3',
  'dabf3130-4570-4fef-86f1-d28b71c421f3',
  '25ba1dc0-23c3-4943-b7fd-ba797e8dcf11',
  'b326fdd0-51b2-40a4-a312-bac271920cc5'],
 'embeddings': array([[-0.00268021, -0.00010323,  0.02815653, ..., -0.01501936,
          0.00590092, -0.01164922],
        [ 0.00092799, -0.00476   ,  0.0124662 , ..., -0.01731381,
          0.00075886,  0.00296567],
        [-0.02714536,  0.00885395,  0.02699314, ..., -0.02592762,
          0.00900617, -0.01999116],
        [-0.01810451,  0.01281202,  0.0347942 , ..., -0.03034012,
         -0.00595078,  0.00521716]], shape=(4, 1536)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah i